# 🇵🇰 Pakistani Social Media Content Detective
**Business question:** *What type of content gets Pakistani audiences to engage, and what should a Pakistani brand do next?*

This notebook walks through the project. All heavy lifting lives in `src/` so the notebook stays readable.
Run it top to bottom (Runtime → Run all). Part A uses real Pakistani Roman Urdu data; Part B needs a Pakistani engagement file (see the README).

## 0. Setup (Colab)
Replace the URL with **your team's GitHub repo**.

In [ ]:
# --- Colab only: clone your repo and install requirements ---
# !git clone https://github.com/<YOUR_USERNAME>/<YOUR_REPO>.git
# %cd <YOUR_REPO>
# !pip install -q -r requirements.txt
import sys, json, pandas as pd
sys.path.insert(0, '.')
pd.set_option('display.max_colwidth', 120); pd.set_option('display.width', 200)

## 1. Run the whole pipeline
Downloads the public Roman Urdu dataset, cleans it, trains the model, analyses themes and writes `reports/`. Takes ~3 minutes.

In [ ]:
!python -m src.run_all

## 2. Data cleaning
Every row we dropped is accounted for.

In [ ]:
print(json.dumps(json.load(open('data/processed/cleaning_log.json')), indent=2))

## 3. Sentiment model
Four models compared with 5-fold cross-validation on the training set only; the best was tuned and scored **once** on a held-out test set.

In [ ]:
from IPython.display import Image, display
m = json.load(open('reports/metrics.json'))
print({k: m[k] for k in ['model','best_param','n_train','n_test','test_accuracy','test_macro_f1','baseline_test_macro_f1']})
display(Image('reports/figures/01_model_comparison.png'))
display(Image('reports/figures/02_confusion_matrix.png'))
display(Image('reports/figures/03_top_words_per_class.png'))

### Where the model is wrong
Real mistakes from the test set - read them, they explain the limits of the model.

In [ ]:
pd.read_csv('reports/misclassified_examples.csv').head(15)

## 4. Which topics drive negativity?
Theme tagging uses a transparent topic-neutral keyword list. Read the README section *A correction we made during development* - an earlier version of this lexicon was circular.

In [ ]:
display(Image('reports/figures/04_theme_negativity.png'))
pd.read_csv('reports/theme_report.csv').round(3)

## 5. From insight to action: AI-assisted recommendations

In [ ]:
from IPython.display import Markdown
Markdown(open('reports/recommendations_roman_urdu_corpus.md', encoding='utf-8').read())

### Try it on a brand's own comments
Swap in your own CSV (any file with a text column).

In [ ]:
from src import recommend
Markdown(recommend.run('data/sample/demo_brand_comments.csv', 'comment'))

## 6. Part B - engagement detective
Needs a **Pakistani engagement file** (likes / shares / comments per post). Map your column names with `--map`.
The cell below is commented out until you add your file to `data/raw/`.

In [ ]:
# !python -m src.engagement --input data/raw/YOUR_FILE.csv --country Pakistan --tz Asia/Karachi \
#     --map likes=like_count,shares=share_count,comments=comment_count,text=caption --tag pakistan
# Markdown(open('reports/engagement_summary_pakistan.md').read())

## 7. Limitations (read before presenting)
1. Training data mixes e-commerce reviews with news/politics/general chatter and has no brand column.
2. The model tends to label *factual* complaints (late delivery, refund pending) as Neutral.
3. Theme keywords are approximate; the team should label `reports/theme_audit_to_label.csv` to measure precision.
4. No theme is significant after multiple-comparison correction in this corpus.
5. Part B has no Pakistani engagement results until a Pakistani engagement dataset is added.